# DGD Test Inference (Algorithm 2)

Loads the frozen `best/` decoder $f_\theta$ + GMM prior and optimizes fresh latents for the held-out test split -- the first real use of `test_loader` in this codebase. $\theta$ and the GMM stay fixed; only $z$ is optimized:

$$
\hat z_i = \arg\min_{z} \; \|f_\theta(\tilde z) - x_i\|_2^2 \;-\; \lambda \log p_{\text{GMM}}(\tilde z), \qquad m = 1, \ldots, M
$$

with a reconstruction-only warm-up for the first $M_0$ steps (`prior_warmup_steps`) before the GMM term is added, and the same noise-injection $\tilde z = z + \epsilon$ used at training time (own `training.inference.latent_noise_*` schedule, annealed over steps instead of epochs) -- since the decoder and GMM were themselves trained under noise, evaluating against a clean $z$ would shift the operating point away from what they were optimized for.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn.functional as F

from omegaconf import OmegaConf, open_dict
from hydra import initialize, compose

current_dir = Path.cwd()
if 'notebooks' in current_dir.parts:
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

from src.data import create_dataloaders, collect_all_labels, collect_class_samples
from src.models import RepresentationLayer, ConvDecoder
from src.utils import setup_device, set_random_seed, setup_cuml_acceleration
from src.utils.checkpoint import load_checkpoint
from src.visualization import generate_inference_figures

device = setup_device(verbose=True)
set_random_seed(seed=42, device=device)
setup_cuml_acceleration(verbose=True)


In [ ]:
with initialize(version_base=None, config_path="../config"):
    config = compose(config_name="config")

config.data.root_dir = str(project_root / "data")
config.paths.experiments_dir = str(project_root / "experiments")

experiments_dir = Path(config.paths.experiments_dir)
candidates = sorted(
    p for p in experiments_dir.glob(f"*_{config.experiment_name}")
    if (p / "models" / "best").is_dir()
)
assert candidates, (
    f"No completed training runs found under {experiments_dir} matching "
    f"*_{config.experiment_name} (looked for a models/best/ subfolder). "
    "Run dgd_training_demo.ipynb first."
)
run_dir = candidates[-1]  # newest, since the timestamp prefix sorts lexicographically
print(f"Using training run: {run_dir}")

# hydra's compose() returns a struct-mode config: models_dir isn't in
# config.yaml's paths: block (only experiments_dir is, since it's resolved
# per-run rather than being a static default), so assigning it requires
# open_dict() to temporarily allow a new key -- a plain assignment would
# raise ConfigAttributeError: Key 'models_dir' is not in struct.
with open_dict(config):
    config.paths.models_dir = str(run_dir / "models")

# Guard: make sure this notebook's config matches the config actually used
# during training, so `test_loader` below is exactly the held-out split
# carved out at training time -- not a silently different split produced by
# a config.yaml that has since changed (random_seed, subset fraction, or
# split ratios).
trained_cfg = OmegaConf.load(run_dir / "config.yaml")
assert config.random_seed == trained_cfg.random_seed, (
    f"random_seed differs from the training run ({config.random_seed} vs {trained_cfg.random_seed}); "
    "the test split would not match the one carved out during training."
)
for key in ['total_subset_fraction', 'val_split', 'test_split']:
    assert config.data[key] == trained_cfg.data[key], (
        f"data.{key} differs from the training run ({config.data[key]} vs {trained_cfg.data[key]}); "
        "the test split would not match the one carved out during training."
    )

# Re-derive the same 3-way split independently (same random_seed as training),
# so `test_loader` here is exactly the held-out split carved out during training
# and never optimized against.
train_loader, val_loader, test_loader, class_names = create_dataloaders(config)
print(f"Test loader: {len(test_loader)} batches, {len(test_loader.dataset)} samples")


In [ ]:
def decoder_factory():
    return ConvDecoder(
        latent_dim=trained_cfg.model.representation.n_features,
        hidden_dims=trained_cfg.model.decoder.hidden_dims,
        output_channels=trained_cfg.model.decoder.output_channels,
        output_size=trained_cfg.model.decoder.output_size,
        activation=trained_cfg.model.decoder.activation,
        final_activation=trained_cfg.model.decoder.final_activation,
        dropout_rate=trained_cfg.model.decoder.dropout_rate,
        init_size=trained_cfg.model.decoder.init_size,
    )

best_dir = Path(config.paths.models_dir) / "best"
checkpoint = load_checkpoint(best_dir, decoder_factory, device=device)

decoder = checkpoint['decoder']
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

gmm = checkpoint['gmm']
if gmm is None:
    raise RuntimeError(
        f"No fitted GMM found in {best_dir} -- was training.first_epoch_gmm ever reached "
        "during training? Re-run dgd_training_demo.ipynb first."
    )

meta = checkpoint['metadata']
print(f"Loaded frozen decoder + GMM from {best_dir} "
      f"(best_epoch={meta.get('best_epoch')}, best_val_loss={meta.get('best_val_loss'):.4f})")


In [ ]:
# Algorithm 2, line 1: initialize a fresh representation layer for the new
# (held-out) data, using the same generic init distribution training used for
# Z_0 -- not a GMM-sample init, matching how the codebase already initializes
# val_rep alongside rep in DGDTrainer._create_model_components.
#
# Uses trained_cfg (the run's own saved config), not the live config: this
# must match the representation dimensionality/distribution the checkpoint's
# decoder and GMM were actually trained with, same reasoning as decoder_factory
# above -- the live config.yaml can have since changed (e.g. n_features).
model_config = trained_cfg.model
distribution = model_config.representation.distribution

if distribution == 'pca':
    print("Representation distribution is 'pca'; using 'normal' for the test layer "
          "instead (matches how val_rep is initialized under PCA during training).")
    test_distribution = 'normal'
    dist_params = {}
else:
    test_distribution = distribution
    dist_params = OmegaConf.to_container(
        model_config.representation.get('dist_params', {}), resolve=True
    )

test_rep = RepresentationLayer(
    dim=model_config.representation.n_features,
    n_samples=len(test_loader.dataset),
    dist=test_distribution,
    dist_params=dist_params,
    device=device,
)
print(f"Initialized test representation layer: {test_rep.n_rep} samples x {test_rep.dim} dims")

# Fail fast, not three cells later inside the optimization loop: if this
# doesn't match, `test_rep` is left over from a previous kernel execution of
# this cell (e.g. only cell 3 and cell 5 were re-run after an edit here) --
# re-run this cell, then cell 5, in that order, in this kernel.
assert test_rep.dim == decoder.decoder_input.in_features, (
    f"test_rep is {test_rep.dim}-dim but decoder expects {decoder.decoder_input.in_features}-dim input. "
    "test_rep is almost certainly stale from an earlier run of this cell -- re-run this cell "
    "(cell 4), then cell 5, in order, in this kernel."
)

In [ ]:
# Algorithm 2, lines 2-5: optimize test_rep alone against the frozen decoder+GMM.
# M0 (prior_warmup_steps) < M (epochs): reconstruction-only warm-up before the
# GMM prior term is added, so a fresh z isn't dominated by the prior gradient
# before it has any reconstruction signal to work with.
import time
from datetime import timedelta

from src.utils.schedules import cosine_noise_schedule

# Fail fast, not partway through the loop below: catches test_rep/decoder
# left over from separate, out-of-sync kernel executions of cells 3 and 4
# (e.g. only one of them re-run after an edit) -- if this fires, re-run
# cell 3, then cell 4, then this cell, in that order, in this kernel.
assert test_rep.dim == decoder.decoder_input.in_features, (
    f"test_rep is {test_rep.dim}-dim but decoder expects {decoder.decoder_input.in_features}-dim input. "
    "test_rep and decoder came from out-of-sync cell executions -- re-run cell 3, then cell 4, "
    "then this cell, in order, in this kernel."
)

lr_config = config.training.lr_scheduler
rep_config = config.training.optimizer.representation

# Resolve the representation LR the same way DGDTrainer._create_optimizers
# does: lr_scheduler.base_lr_representation overrides optimizer.representation.lr
# when the scheduler is enabled, since CosineAnnealingLR has no base_lr
# argument of its own -- it reads its starting LR straight off the optimizer.
test_rep_lr = rep_config.lr
if lr_config.get('enabled', False) and lr_config.get('base_lr_representation', None) is not None:
    test_rep_lr = lr_config.base_lr_representation

test_optimizer = torch.optim.AdamW(
    test_rep.parameters(),
    lr=test_rep_lr,
    betas=tuple(rep_config.betas),
    eps=rep_config.eps,
    weight_decay=rep_config.weight_decay,
    amsgrad=rep_config.get('amsgrad', False),
)

M = config.training.inference.epochs
M0 = config.training.inference.prior_warmup_steps
assert M0 < M, "training.inference.prior_warmup_steps must be < training.inference.epochs"

if lr_config.get('enabled', False):
    test_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        test_optimizer,
        T_max=M,
        eta_min=lr_config.final_lr_representation,
    )
else:
    test_scheduler = None

lambda_gmm = config.training.lambda_gmm
n_test = len(test_loader.dataset)

# Inference has its own noise schedule, independent of training's
# latent_noise_enabled/_start/_end -- same cosine-anneal formula, mapped onto
# step index m instead of epoch.
latent_noise_enabled = config.training.inference.get('latent_noise_enabled', False)
noise_start = config.training.inference.get('latent_noise_start', 1.0)
noise_end = config.training.inference.get('latent_noise_end', 0.01)

print(f"Optimizing {n_test} test representations for {M} steps (prior warm-up: {M0} steps)...")

step_history = {'loss': [], 'recon': [], 'gmm': [], 'noise': []}

# In-memory z snapshots every 50 steps, for the noise-comparison figures
# generated later (cell 7) -- inference has no per-step disk checkpointing
# (unlike training's every-50-epoch checkpoints), so these stay in memory
# for the duration of this kernel session.
noise_snapshots = []

best_loss = float('inf')
best_recon = float('inf')
best_gmm = float('inf')
step_times = []

for m in range(1, M + 1):
    step_start_time = time.time()
    test_optimizer.zero_grad()

    if latent_noise_enabled:
        noise_scale_m = cosine_noise_schedule(m, M, noise_start, noise_end)
    else:
        noise_scale_m = 0.0

    total_loss = 0.0
    total_recon = 0.0
    total_gmm = 0.0

    for index, x, _ in test_loader:
        x, index = x.to(device), index.to(device)

        z = test_rep(index)

        if noise_scale_m > 0:
            z = z + torch.randn_like(z) * noise_scale_m

        y = decoder(z)
        recon_loss = F.mse_loss(y, x, reduction='sum')

        if m >= M0:
            gmm_error = -lambda_gmm * torch.sum(gmm.score_samples(z))
            loss = recon_loss + gmm_error
        else:
            gmm_error = torch.tensor(0.0, device=device)
            loss = recon_loss

        loss.backward()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_gmm += gmm_error.item()

    test_optimizer.step()
    if test_scheduler is not None:
        test_scheduler.step()

    avg_loss = total_loss / n_test
    avg_recon = total_recon / n_test
    avg_gmm = total_gmm / n_test

    step_history['loss'].append(avg_loss)
    step_history['recon'].append(avg_recon)
    step_history['gmm'].append(avg_gmm)
    step_history['noise'].append(noise_scale_m)

    if avg_loss < best_loss:
        best_loss = avg_loss
    if avg_recon < best_recon:
        best_recon = avg_recon
    if m >= M0 and avg_gmm < best_gmm:
        best_gmm = avg_gmm

    if m % 50 == 0 or m == M:
        with torch.no_grad():
            noise_snapshots.append({
                'step': m,
                'z': test_rep.z.detach().clone(),
                'noise_scale': noise_scale_m,
            })

    step_times.append(time.time() - step_start_time)

    if m % max(1, M // 10) == 0 or m == M:
        avg_step_time = sum(step_times) / len(step_times)
        remaining_time_str = str(timedelta(seconds=int((M - m) * avg_step_time)))
        elapsed_time_str = str(timedelta(seconds=int(sum(step_times))))
        lr_rep = test_optimizer.param_groups[0]['lr']
        gmm_str = f"{avg_gmm:.4f} (B: {best_gmm:.4f})" if m >= M0 else "0.0000"

        print(f"Step {m}/{M} [Elapsed: {elapsed_time_str}, Remaining: {remaining_time_str}, LR: Rep={lr_rep:.2e}, Noise={noise_scale_m:.4f}]")
        print(f"       - Loss: {avg_loss:.4f} (B: {best_loss:.4f}), Recon: {avg_recon:.4f} (B: {best_recon:.4f}), GMM: {gmm_str}")

print("Test representation optimization complete.")


In [ ]:
from datetime import datetime

inference_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
inference_dir = run_dir / "inference" / inference_timestamp
inference_dir.mkdir(parents=True, exist_ok=True)

OmegaConf.save(config, str(inference_dir / "config.yaml"))

test_rep.save(str(inference_dir / "test_representation.pt"))
print(f"Saved optimized test representations to {inference_dir / 'test_representation.pt'}")


In [ ]:
figures_dir = inference_dir / "figures"

test_labels = collect_all_labels(test_loader)

sample_data = collect_class_samples(test_loader, n_per_class=5, n_classes=len(class_names))

step_history_totals = step_history

# Fail fast: noise_snapshots comes from cell 5's optimization loop, not
# reloaded from disk -- if cell 5 hasn't run in this kernel (or a stale
# test_rep/decoder pairing slipped through), this catches it here instead of
# a confusing shape mismatch inside plot_noise_comparison below.
assert noise_snapshots, (
    "noise_snapshots is empty -- re-run cell 5 (the optimization loop) in this kernel before this cell."
)
assert noise_snapshots[-1]['z'].shape[1] == test_rep.dim, (
    f"noise_snapshots dim ({noise_snapshots[-1]['z'].shape[1]}) doesn't match test_rep.dim ({test_rep.dim}) -- "
    "noise_snapshots is stale from an earlier run of cell 5 -- re-run cell 5, then this cell, in order, in this kernel."
)

test_ami, test_ari = generate_inference_figures(
    figures_dir=figures_dir,
    decoder=decoder,
    gmm=gmm,
    test_rep=test_rep,
    test_labels=test_labels,
    class_names=class_names,
    sample_data=sample_data,
    step_history=step_history_totals,
    device=device,
    noise_snapshots=noise_snapshots,
)

print(f"Test AMI: {test_ami:.4f}, Test ARI: {test_ari:.4f}")
print(f"Figures written to {figures_dir}")
